## Hooks
React Hooks let us use state and other React features without writing a class.

## useEffect
`useEffect` hook lets us perform side effects in function components like data fetching, setting up a subscription, and manually changing the DOM in React components. Effect hook is triggered after rendering.

In [ ]:
import { useState, useEffect } from 'react';

function Component() {
  const [count, setCount] = useState(0);

  useEffect(() => {
      document.title = `You clicked ${count} times`;  
  });

  return (
    <div>
      <p>You clicked {count} times</p>
      <button onClick={() => setCount(count + 1)}>
        Click me
      </button>
    </div>
  );
}

Function inside the above `useEffect` hook runs after every render once the browser has painted the DOM changes. In the above scenario:
- on initial mount, the component has `count=0` which is rendered in the browser. After that the `useEffect` runs and changes document title with `count=0`.
- user clicks the button, setting `count=1`. This triggers a render. Browser paints the component on screen with text `count=1`. After that the `useEffect` hook runs and changes the document title to `count=1`.

From inside the hook, we can also return a cleanup function:

In [ ]:
useEffect(() => {
    return () => {
        // Cleanup code
    }
})

Cleanup function runs cleanup from the previous render. This means that it doesn't run on initial mount. Example:

In [ ]:
useEffect(() => {
    // Subscribe to chat API for status change
    ChatAPI.subscribeToFriendStatus(props.friend.id, handleStatusChange);
    
    // Unsubscribe
    return () => {
      ChatAPI.unsubscribeFromFriendStatus(props.friend.id, handleStatusChange);
    };
    
});

Consider the following set of events:
1. Component mounted with props set as `{ friend: { id: 1 } }`, effect executed after render is committed:

In [ ]:
ChatAPI.subscribeToFriendStatus(1, handleStatusChange);

2. Prop changed to `{ friend: { id: 2 } }`, effect executed after render is committed:

In [ ]:
ChatAPI.unsubscribeFromFriendStatus(1, handleStatusChange);
ChatAPI.subscribeToFriendStatus(2, handleStatusChange);

3. Prop changed to `{ friend: { id: 3 } }`, effect executed:

In [ ]:
ChatAPI.unsubscribeFromFriendStatus(2, handleStatusChange);
ChatAPI.subscribeToFriendStatus(3, handleStatusChange);

4. Finally, when the component is unmounted:

In [ ]:
ChatAPI.unsubscribeFromFriendStatus(3, handleStatusChange);

Another example:

In [ ]:
function WindowWidth() {
	const [width, setWidth] = useState(window.innerWidth)

	useEffect(() => {
		const handler = () => setWidth(window.innerWidth)

		// This event listener is bound to the window, therefore it would persist even after
		// the component is long gone, thereby creating memory leak
		window.addEventListener("resize", handler)

		// Which is why we need to remove it
		return () => {
			window.removeEventListener("resize", handler)
		}
	});

	return (
		<><p>Window width is: {width}</p></>
	)
}

![useEffect flow](./images/useeffect_seq.png)

As with useState, we can have multiple `useEffect` calls. They are executed in the order they appear.

In the previous codes, `useEffect` runs on every render. However, we can pass it an array of dependencies which React would compare (against value from previous render) to determine whether or not `useEffect` should be run. `useEffect` would then run on initial render and when one of the dependencies change: 

In [ ]:
// Run the below effect only if count has changed
// To determine if count has changed,`Object.is` method
// is used and compared to older value. This is similar
// to == operator.
useEffect(() => {
  document.title = `You clicked ${count} times`;
}, [count]);

The below `useEffect` hook would run every time:

In [ ]:
useEffect(() => {
  document.title = `You clicked ${count} times`;
}, [{ count: 0 }]);

## useReducer
Reducers help us to consolidate state update logic outside of the component in reducer function. Reducers offer an alternative to `useState`. Instead of setting state, we dispatch actions. Lets say we have a event handler that handles adding and removing task:

In [ ]:
function handleTaskAdd(task) {
    setTasks([
        ...tasks,
        task
    ]);
}

function handleTaskRemove(task) {
    setTasks(tasks.filter(t => t == task));
}

When using reducers, we would instead:

In [ ]:
// The object passed is called action
// action doesn't have a specific schema, but it
// should describe what happened well
function handleTaskAdd(task) {
    dispatch({
        type: 'add',
        task: task
    })
}

function handleTaskRemove(task) {
    dispatch({
        type: 'remove',
        task: task
    })
}

Next we write reducer function:

In [ ]:
// Reducer function must be pure
function taskReducer(tasks, action) { // argument is the current state and the action
    if (action.type === 'add') {
        return [
            ...tasks,
            action.task
        ]
    } else if (action.type === 'remove') {
        return tasks.filter(t => t == action.task)
    } else {
        throw Error("Unknown action);
    }
}

Now inside the component, we call `useReducer`:

In [ ]:
const [tasks, dispatch] = useReducer(taskReducer, ['Initial task']);

What about setting state using a function, like in `useState(() => expensiveOperation())`? `useReducer` uses a different technique:

In [ ]:
function init(initialCount) {
    return { count: initialCount * 2 };
}

function reducer(state, action) {
    switch (action.type) {
        case "increment":
            return { count: state.count + 1 };
        default:
            return state;
  }
}

function Counter({ initialCount }) {
    const [state, dispatch] = useReducer(reducer, initialCount, init); // Second parameter will be passed as argument to init function
    // ....

If we want `init` function to not depend upon any parameter:

In [ ]:
const [state, dispatch] = useReducer(reducer, null, () => return 10);

## useRef
Lets us store some information in a component without trigerring a re-render everytime we change it.

In [ ]:
const ref = useRef(0);

/*
Returns
{
    current : 0
}
*/

Unlike state, ref can be freely read and written to.

In [ ]:
export default function Counter() {
    let countRef = useRef(0);
    
    function handleClick() {
        // This doesn't re-render the component!
        countRef.current = countRef.current + 1;
    }
    
    // Pressing the button doesn't update You clicked 0 times
    return (
        <button onClick={handleClick}>
            You clicked {countRef.current} times
        </button>
    );
}

We can think of `useRef` as `useState` without the setter function. Infact its implementation looks like:

In [ ]:
function useRef(initialValue) {
    const [ref, unused] = useState({ current: initialValue });
    return ref;
}

A common use of ref is to point to a DOM element:

In [ ]:
export default function Input() {
  const inputRef = useRef(null);

  function handleClick() {
    inputRef.current.focus();
  }

  return (
    <>
      <input ref={inputRef} />
      <button onClick={handleClick}>
        Focus the input
      </button>
    </>
  );
}